In [10]:
# Repository path setup. This lets the notebook run from either the repo root or the notebooks/ folder.
from pathlib import Path
import os

_REPO_ROOT = Path.cwd()
if not (_REPO_ROOT / "data").exists() and (_REPO_ROOT.parent / "data").exists():
    _REPO_ROOT = _REPO_ROOT.parent
os.chdir(_REPO_ROOT)
# print(f"Working directory: {_REPO_ROOT}")

# CTN-0094 dataset generation (static_timeSeries_new.csv)

This notebook builds the analysis-ready table from the public CTN-0094 release.

In [2]:
import pyreadr # pyreadr does not support numpy 2.x
import os
from matplotlib import pyplot as plt

# Define the directory containing the .rda files
directory = 'data/source/public.ctn0094data/data/'

# Initialize an empty dictionary to store the DataFrames
dataframes = {}

# Iterate over all files in the directory
for filename in os.listdir(directory):
    if filename.endswith('.rda'):
        file_path = os.path.join(directory, filename)

        # Read the .rda file
        result = pyreadr.read_r(file_path)

        # Extract data into DataFrame
        for key in result.keys():
            dataframes[filename[:-4]] = result[key]

# Print the names and first rows of the DataFrames loaded
print("Loaded DataFrames:", dataframes.keys())

import warnings
warnings.filterwarnings('ignore')

Loaded DataFrames: dict_keys(['withdrawal_pre_post', 'fagerstrom', 'asi', 'sex', 'derived_inductDelay', 'site_masked', 'rbs', 'everybody', 'all_drugs', 'meta_study_length', 'meta_substance_groups_uds', 'pain', 'treatment', 'uds_temp', 'screening_date', 'randomization', 'derived_visitImputed', 'demographics', 'tlfb', 'derived_weeklyTLFBPattern', 'qol', 'first_survey', 'derived_raceEthnicity', 'visit', 'psychiatric', 'withdrawal', 'detox', 'uds', 'derived_weeklyOpioidPattern', 'rbs_iv'])


In [3]:
import warnings
warnings.filterwarnings("ignore")

In [4]:
everybody = dataframes['everybody']
treatment = dataframes['treatment']
all_drugs = dataframes['all_drugs']
uds = dataframes['uds']
uds_temp = dataframes['uds_temp']
visit = dataframes['visit']
tlfb = dataframes['tlfb']
randomization = dataframes['randomization']
withdrawal = dataframes['withdrawal']
screening_date = dataframes['screening_date']
site_masked = dataframes['site_masked']

In [5]:
opioid_substances = ['Morphine', 'Oxycodone', 'Fentanyl', 'Opioid', 'Heroin', 'Opium', 'Buprenorphine', 'Methadone',
                     'Hydromorphone', 'Hydrocodone', 'Tramadol', 'Propoxyphene', 'Oxymorphone', 'Codeine', 'Merperidine', 'Nalbuphine']
prescribed_opioids = {
    "Outpatient BUP + EMM": "Buprenorphine",
    "Outpatient BUP + SMM": "Buprenorphine",
    "Outpatient BUP": "Buprenorphine",
    "Inpatient BUP": "Buprenorphine",
    "Methadone": "Methadone",
    "Inpatient NR-NTX": "Naltrexone"
}
wk_step = 7
def get_Return_By12Wk(miss_flag = 'UDS',randomization = randomization, uds = uds, everybody = everybody, uds_temp = uds_temp, visit=visit):
    patient_ids = randomization.who.unique()
    outcomes = {} # 4 consecutive opioid positive weeks between 3 and 12 week after randomization
    return_to_use = {}
    treatment_entry = {}
    prescribed_opioid_all = {}
    cnt_pos, cnt_neg = 0, 0
    for pid in patient_ids:
        proj_id = everybody[everybody.who==pid]['project'].values[0]
        ## here exclude the prescribed opioid for the patient in UDS
        randomization_pid = randomization[(randomization.who==pid)&(~randomization['when'].isna())] # for the current patient, drop the randomization record i the date ("when") is na
        if proj_id == '30': # remove Phase I in project 30 to be consistent with JAMA Psychiatry paper
            randomization_pid = randomization_pid[randomization_pid.which != '1']
        if randomization_pid.shape[0] == 0:
            continue
        random_group = randomization_pid.treatment.values[0]
        prescribed_opioid = prescribed_opioids[random_group]


        ## Detect correct range of study for different patients
        # There is no need to separately process project 30 as the phase I has been removed
        random_date = int(randomization_pid.when.values[0])
        range_start = random_date
        range_stop = 24*wk_step + range_start


        prescribed_opioid_all[pid] = prescribed_opioid
        treatment_entry[pid] = random_date
        outcome = ''
        accum_cnt = 0 # count consecutive positive UDS week
        return_to_use[pid] = 0

        output_range_start = random_date+wk_step*3
        output_range_stop = random_date+wk_step*12

        for i in range(range_start,range_stop,wk_step):
            start = i+1 # >=
            stop = min(i+wk_step+1,range_stop+1) # <

            visit_exist_pid = visit[(visit.who==pid)&(visit.when>=start)&(visit.when<stop)&((visit.what=='visit')|(visit.what=='final'))]
            uds_exist_pid = uds_temp[(uds_temp.who==pid)&(uds_temp.when>=start)&(uds_temp.when<stop)&(uds_temp.was_temp_ok=='1')]

            exist_num = uds_exist_pid.shape[0]
            if miss_flag == 'visit':
                exist_num += visit_exist_pid.shape[0]


            uds_pid = uds[(uds.who==pid)&(uds.when>=start)&(uds.when<stop)&(uds.what!=prescribed_opioid)] # extract

            if exist_num==0: # no UDS result at all
                outcome+='o'
                if (i >= output_range_start) and (i < output_range_stop):
                    accum_cnt += 1
            else:
                ## consider all days in both visit and uds if 'visit' is set
                days_pid = uds_exist_pid.when.to_list()
                if miss_flag == 'visit':
                    days_pid += visit_exist_pid.when.to_list()
                days_pid = sorted(list(set(days_pid)))

                outcome_list = []
                for day_pid in days_pid:
                    uds_result_pid = uds_pid[uds_pid.when == day_pid].what.values
                    if len(set(uds_result_pid).intersection(set(opioid_substances))) >= 1: # positive for nonprescribed opioid
                        outcome_list.append('+')
                    else: # negative UDS result
                        outcome_list.append('-')

                if len(set(outcome_list)) > 1:
                    outcome+='*'
                    if (i >= output_range_start) and (i < output_range_stop):
                        accum_cnt += 1
                elif len(set(outcome_list)) == 0:
                    outcome+='-'
                    if (i >= output_range_start) and (i < output_range_stop):
                        accum_cnt = 0
                elif outcome_list[0] == '+':
                    outcome+='+'
                    if (i >= output_range_start) and (i < output_range_stop):
                        accum_cnt += 1
                else:
                    outcome+='-'
                    if (i >= output_range_start) and (i < output_range_stop):
                        accum_cnt = 0

            if accum_cnt >= 4:
                return_to_use[pid] = 1

        outcomes[pid] = outcome
        if return_to_use[pid] == 1:
            cnt_pos += 1
        else:
            cnt_neg += 1

    print(cnt_pos,cnt_neg)

    return outcomes, return_to_use, treatment_entry, prescribed_opioid_all
outcomes_visit, return_to_use_visit, treatment_entry_visit, prescribed_opioids_visit = get_Return_By12Wk(miss_flag='visit')

1083 1116


In [6]:
import pandas as pd
def get_predictors_full(ctn94wk, treatment_entry, prescribed_opioid_all, dataframes=dataframes):
    # first extract the list of patient ids
    ctn94wk[ctn94wk.isna()] = -1
    map_dic = {1:'+',0:'-',-1:'o'}
    pid = sorted(ctn94wk.who.tolist())


    def get_label(ctn94wk, pid):
        label_list = []
        for id in pid:
            val = (ctn94wk[ctn94wk.who==id].values[0][3:12]!=0).astype(int)
            label_val = 0
            for i_tmp in range(6):
                if val[i_tmp:(i_tmp+4)].sum()==4:
                    label_val = 1
                    break
            label_list.append(label_val)

        return label_list

    label = get_label(ctn94wk, pid)

    dataframes['demographics'] = dataframes['demographics'].sort_values('who').reset_index(drop=True)

    demo_df = dataframes['demographics'].loc[dataframes['demographics'].who.isin(pid),['age','is_hispanic','race','job','is_living_stable','education','marital','is_male']]
    age,is_hispanic,race,job,is_living_stable,education,marital,is_male = [demo_df[col].tolist() for col in demo_df.columns]
    race = [ele if (ele == 'Black') or (ele == 'White') else 'Other' for ele in race]
    race_dic = {'Black': 1, 'White': 2, 'Other': 3}
    race = [race_dic[ele] for ele in race]
    is_male = [1 if ele == 'Yes' else 0 for ele in is_male]
    dataframes['everybody'] = dataframes['everybody'].sort_values('who').reset_index(drop=True)
    project = dataframes['everybody'].loc[dataframes['everybody'].who.isin(pid),'project'].values.tolist()

    Randomize = dataframes['randomization'][['who','treatment']].drop_duplicates().sort_values('who').reset_index(drop=True)
    treat = [Randomize.loc[Randomize.who==id,'treatment'].values[0] for id in pid]


    treatment_group = []
    for proj, trt in zip(project, treat):
        if proj == '30':
            treatment_group.append(3)
        elif trt == 'Inpatient NR-NTX':
            treatment_group.append(5)
        elif proj == '51':
            treatment_group.append(4)
        elif trt == 'Methadone':
            treatment_group.append(2)
        else:
            treatment_group.append(1)

    # Extract Treatment Data for Use
    Treatment = dataframes['treatment']

    # per-patient list of [amount, when]
    treat_record = [
        Treatment.loc[Treatment.who == id, ['amount', 'when']].values.tolist()
        if id in Treatment.who.tolist() else []
        for id in pid
    ]

    # treatment entry days (you already have this dict)
    treatment_entry_days = [treatment_entry[id] if id in treatment_entry else None for id in pid]

    # keep your original binary "any treatment" series
    def convert_treat_record_binary(treat_entry, treat_record):
        treatment_arr = ['0'] * 168
        if (treat_entry is not None) and (len(treat_record) > 0):
            for amt, day in treat_record:  # amt is unused here, we only look at day
                cali_day = int(day - treat_entry + 7)  # we use the past week of the visit as the data, a whole week before the visit day, so it should be 7 rather than 6
                if 0 <= cali_day < len(treatment_arr):
                    treatment_arr[cali_day] = '1'
        return ','.join(treatment_arr)

    treat_weeks = [
        convert_treat_record_binary(entry_day, rec)
        for entry_day, rec in zip(treatment_entry_days, treat_record)
    ]


    prescribed_opioids = {
        "Outpatient BUP + EMM": "Buprenorphine",
        "Outpatient BUP + SMM": "Buprenorphine",
        "Outpatient BUP": "Buprenorphine",
        "Inpatient BUP": "Buprenorphine",
        "Methadone": "Methadone",
        "Inpatient NR-NTX": "Naltrexone"
    }

    canonical_types = ["Buprenorphine", "Methadone", "Naltrexone"]

    # will hold one list per canonical type, each list length = number of patients
    treat_amounts_by_type = {drug: [] for drug in canonical_types}

    # treat is already defined above as the randomization 'treatment' string per patient
    for id_, entry_day, rec, trt_name in zip(pid, treatment_entry_days, treat_record, treat):
        # init per-patient series for each canonical type as zeros
        per_type_arrays = {drug: ['0'] * 168 for drug in canonical_types}

        if (entry_day is not None) and (len(rec) > 0):
            # map randomization treatment to canonical MOUD type, if known
            canonical = prescribed_opioids.get(trt_name, None)

            if canonical in per_type_arrays:
                for amount, day in rec:
                    cali_day = int(day - entry_day + 7)
                    if 0 <= cali_day < 168:
                        per_type_arrays[canonical][cali_day] = str(float(amount))

        # append 168-length comma-joined strings for each MOUD type
        for drug in canonical_types:
            treat_amounts_by_type[drug].append(','.join(per_type_arrays[drug]))


    # rbs_iv -> heroin inject days
    rbs_iv = dataframes['rbs_iv'].sort_values('who').reset_index(drop=True)
    rbs_iv_rec = rbs_iv.loc[rbs_iv.who.isin(pid)&(rbs_iv.heroin_inject_days>0),['who','heroin_inject_days']]
    # heroin_inject_days = [rbs_iv_rec.loc[rbs_iv_rec.who==id,'heroin_inject_days'].values[0] if id in rbs_iv_rec.who.values else 0.0 for id in pid]
    heroin_inject = [1 if id in rbs_iv_rec.who.values else 0 for id in pid]

    # all_drugs -> morphine- and cocaine-positive UDS results, before the treatment entry day
    uds_all_drug_list = dataframes['uds'].what.unique().tolist()
    uds_drugs = {drug:[] for drug in uds_all_drug_list}

    tlfb_drugs = {'Heroin': [],'THC': [], 'Alcohol': [], 'Cocaine': [], 'Methadone': [], 'Amphetamine': []}

    for id in pid:
        entry_day = treatment_entry[id]

        tlfb_start_day = entry_day - 7*4

        for drug in tlfb_drugs.keys():
            tlfb_id_tab = dataframes['tlfb'][(dataframes['tlfb'].who==id)&(dataframes['tlfb'].what==drug)&(dataframes['tlfb'].when>tlfb_start_day)&(dataframes['tlfb'].when<=entry_day)]
            tlfb_drugs[drug].append(tlfb_id_tab.shape[0])


        for drug in uds_drugs.keys():
            uds_id = dataframes['all_drugs'][(dataframes['all_drugs'].who==id)&(dataframes['all_drugs'].what==drug)&(dataframes['all_drugs'].when<=entry_day)&(dataframes['all_drugs'].when>=0)]
            if uds_id.shape[0] > 0:
                uds_drugs[drug].append(1)
            else:
                uds_drugs[drug].append(0)

        prescribed_opioid = prescribed_opioid_all[id]


    # first three week test. non-prescribed UDS >> Opioids, Oxycodone, Heroin, non-prescribed Methadone, non-prescribed Buprenorphine,
    weekly_drugs = {}
    name_dic = {'o':'Opioid','amp':'Amphetamine', 'ben':'Benzodiazepine', 'bup':'Buprenorphine', 'can':'Cannabis', 'coc':'Cocaine', 'her':'Heroin', 'met':'Methadone', 'methamp':'Methamphetamine', 'oxy':'Oxycodone', 'propoxy':'Propoxyphene'}
    for i in range(0,ctn94wk.shape[1]-1,24):
        keywd = ctn94wk.columns[i][:-6]

        for id in pid:
            for wk in range(24):
                wk_kw = f'{name_dic[keywd]}_week{wk}'
                if not wk_kw in weekly_drugs:
                    weekly_drugs[wk_kw] = [ctn94wk[ctn94wk.who==id].values[0][i+wk]]
                else:
                    weekly_drugs[wk_kw].append(ctn94wk[ctn94wk.who==id].values[0][i+wk])

    # existing core columns
    res = [pid, label, treatment_group, treat_weeks, age, race, is_male, heroin_inject]
    res_names = ['who', 'return_to_use', 'treatment_group', 'treat_wks',
                 'age', 'race', 'is_male', 'heroin_inject']

    for drug in canonical_types:
        col_name = f"treat_{drug}_amt"
        res.append(treat_amounts_by_type[drug])
        res_names.append(col_name)


    for drug in tlfb_drugs.keys():
        res.append(tlfb_drugs[drug])
        res_names.append(f'TLFB_{drug}')

    for drug in uds_drugs.keys():
        res.append(uds_drugs[drug])
        res_names.append(f'UDS_{drug}')

    for drug_n in weekly_drugs.keys():
        # drug_n = drug + '_uds'
        res.append(weekly_drugs[drug_n])
        res_names.append(drug_n)

    return res, res_names

ctn94wk = pd.read_excel('data/external/CTN94WKUDS_E_0330_0820.xlsx')
res_visit_full, res_names_visit_full = get_predictors_full(ctn94wk, treatment_entry_visit, prescribed_opioids_visit)

In [7]:

res_df_visit_full = pd.DataFrame(res_visit_full).T
res_df_visit_full.columns = res_names_visit_full

# --- Add masked site ID (for site-shift benchmark splits) ---
site_masked = dataframes.get('site_masked', None)
if site_masked is not None:
    # Identify a site column robustly (different releases may name it differently)
    candidate_cols = ['site_masked', 'site', 'site_id', 'where', 'center', 'node', 'masked_site']
    site_col = next((c for c in candidate_cols if c in site_masked.columns), None)
    if site_col is None:
        non_who_cols = [c for c in site_masked.columns if c != 'who']
        site_col = non_who_cols[0] if len(non_who_cols) else None
    if site_col is not None:
        site_map = site_masked[['who', site_col]].drop_duplicates().rename(columns={site_col: 'site'})
        res_df_visit_full = res_df_visit_full.merge(site_map, on='who', how='left')

# --- Add randomization/treatment-entry day (for time-shift splits) ---
# treatment_entry_visit is a dict {who: randomization_day} created by get_Return_By12Wk above
if 'treatment_entry_visit' in globals():
    res_df_visit_full['rand_day'] = res_df_visit_full['who'].map(treatment_entry_visit)

print(res_df_visit_full.return_to_use.sum())
res_df_visit_full.to_csv('data/processed/static_timeSeries_new.csv', index=None)
res_df_visit_full

1096


,who,return_to_use,treatment_group,treat_wks,age,race,is_male,heroin_inject,treat_Buprenorphine_amt,treat_Methadone_amt,...,Cannabis_week16,Cannabis_week17,Cannabis_week18,Cannabis_week19,Cannabis_week20,Cannabis_week21,Cannabis_week22,Cannabis_week23,site,rand_day
0,3,1,4,"0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...",23.0,1,0,1,"0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...",...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,510007,13
1,4,0,5,"0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...",19.0,2,1,1,"0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...",...,1.0,1.0,1.0,1.0,-1.0,1.0,-1.0,-1.0,510003,1
2,7,1,5,"0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...",33.0,2,0,1,"0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...",...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,510005,5
3,9,1,5,"0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...",25.0,1,0,1,"0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...",...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,510007,15
4,10,1,1,"0,0,0,0,0,0,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,...",29.0,3,0,1,"0,0,0,0,0,0,0,8.0,16.0,24.0,24.0,24.0,24.0,24....","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...",...,0.0,1.0,-1.0,0.0,0.0,1.0,-1.0,1.0,270012,19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2194,3556,0,3,"0,0,0,0,0,0,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,...",56.0,2,1,0,"0,0,0,0,0,0,0,8.0,16.0,16.0,16.0,16.0,16.0,16....","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...",...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,510007,49
2195,3557,1,1,"0,0,0,0,0,0,0,1,1,1,1,1,1,0,0,1,0,0,1,1,1,0,0,...",31.0,2,1,1,"0,0,0,0,0,0,0,4.0,6.0,6.0,6.0,6.0,6.0,0,0,6.0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...",...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,270005,12
2196,3558,1,2,"0,0,0,0,0,0,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,...",27.0,2,1,0,"0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,30.0,50.0,50.0,70.0,70.0,70.0,70...",...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,270001,4
2197,3559,1,5,"0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...",34.0,3,1,1,"0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...",...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0.0,510006,5


In [8]:
def extend_ones(data, window=29):
    """
    Changes the following `window` entries after each '1' in the list to '1'.

    Parameters:
        data (list): List of 0s and 1s.
        window (int): Number of entries to change after each '1'.

    Returns:
        list: Modified list with extended 1s.
    """
    result = data[:]  # Copy the original list
    n = len(data)

    for i in range(n):
        if data[i] != '0':
            for j in range(1, window + 1):
                if i + j < n:  # Ensure we don't go out of bounds
                    result[i + j] = data[i]
    return ','.join(result)

In [9]:
######## Important update: change effective of shot treatment from 1 day to 30 days
######## Just run once
for i in range(res_df_visit_full.shape[0]):
    if res_df_visit_full.loc[i, 'treatment_group'] == 5:
        treat_record = res_df_visit_full.loc[i, 'treat_wks'].split(',')
        res_df_visit_full.loc[i, 'treat_wks'] = extend_ones(treat_record, window=29)
        treat_record = res_df_visit_full.loc[i, 'treat_Naltrexone_amt'].split(',')
        res_df_visit_full.loc[i, 'treat_Naltrexone_amt'] = extend_ones(treat_record, window=29)
res_df_visit_full.to_csv('data/processed/static_timeSeries_new.csv',index=None)